## Método del disparo

El método del disparo consiste en transformar un problema de valores en la frontera (BVP) en un problema de valor inicial (IVP).

En este caso, se conoce:

$$
u(13) = 0.01007, \quad u(20) = 0.00769
$$

pero no se conoce la derivada inicial:

$$
z(13) = u'(13)
$$

La idea es asumir un valor inicial para $z(13)$, resolver el sistema y ajustar iterativamente hasta cumplir la condición en $r = 20$.



In [10]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros
r0 = 13
rf = 20
N = 4
h = (rf - r0)/N

r_vals = np.linspace(r0, rf, N+1)

# Condiciones
u0 = 0.01007
u_target = 0.00769

# Sistema
def f1(r, u, z):
    return z

def f2(r, u, z):
    return -z/r + u/(r**2)

# Integrador (Euler)
def integrar(z0):
    u = np.zeros(N+1)
    z = np.zeros(N+1)
    
    u[0] = u0
    z[0] = z0
    
    for i in range(N):
        r = r_vals[i]
        
        u[i+1] = u[i] + f1(r, u[i], z[i]) * h
        z[i+1] = z[i] + f2(r, u[i], z[i]) * h
    
    return u, z

def shooting_secante(z1, z2, tol=1e-15, max_iter=50):
    for _ in range(max_iter):
        u1, _ = integrar(z1)
        u2, _ = integrar(z2)
        
        y1 = u1[-1]
        y2 = u2[-1]
        
        # fórmula de la diapositiva
        z_new = z1 + (u_target - y1)*(z2 - z1)/(y2 - y1)
        
        u_new, _ = integrar(z_new)
        y_new = u_new[-1]
        
        if abs(y_new - u_target) < tol:
            return z_new
        
        # actualizar
        z1, z2 = z2, z_new
    
    return z_new
z_opt = shooting_secante(-0.01, 0.0)
u_sol, z_sol = integrar(z_opt)

# Solución final
u_sol, z_sol = integrar(z_opt)

from tabulate import tabulate

tabla = []

for i in range(N+1):
    tabla.append([
        i,
        r_vals[i],
        u_sol[i],
        z_sol[i]
    ])

print(tabulate(tabla, 
               headers=["i", "r (cm)", "u(r) (cm)", "z(r)"], 
               floatfmt=".6f"))


  i     r (cm)    u(r) (cm)       z(r)
---  ---------  -----------  ---------
  0  13.000000     0.010070  -0.000553
  1  14.750000     0.009102  -0.000375
  2  16.500000     0.008446  -0.000257
  3  18.250000     0.007997  -0.000175
  4  20.000000     0.007690  -0.000117
